# Diagnose the running load

Tells you what the server is actually doing right now, regardless of which
version of `main.py` / `build_benefits.py` is deployed. Does NOT submit a
new payload - just inspects an existing run.

Three possible verdicts at the end:
1. **Server is making progress** - wait longer, or process fewer plans per run
2. **One plan is stuck** - need to look at Flask logs for that plan
3. **Worker died** - need to restart and resume from checkpoints


## 0. Config

In [ ]:
import json
import requests
from datetime import datetime, timezone

BASE_URL = 'http://127.0.0.1:8080'   # or the Azure App Service URL
LOAD_ID  = '210'                     # the load you started

print(f'Target: {BASE_URL}')
print(f'Load:   {LOAD_ID}')


## 1. Check what version of build_benefits is running

In [ ]:
r = requests.get(f'{BASE_URL}/', timeout=15)
health = r.json()
print(json.dumps(health, indent=2))

server_ver = health.get('build_benefits_ver')
print()
if server_ver == '2026-05-12-sweeper-and-retry-v5':
    print('OK - server is running v5 with checkpoint support')
    HAS_CHECKPOINTS = True
elif server_ver and 'v5' in server_ver:
    print(f'OK - looks like v5 variant: {server_ver}')
    HAS_CHECKPOINTS = True
elif server_ver:
    print(f'WARNING - server is running {server_ver}')
    print('         The new /status endpoint may not exist on this build.')
    HAS_CHECKPOINTS = False
else:
    print('WARNING - server did not report a build version')
    HAS_CHECKPOINTS = False


## 2. Get the lightweight status (no row data)

This is the most useful endpoint for debugging. It returns plan-level progress
without dragging back potentially-thousands of result rows.

In [ ]:
if not HAS_CHECKPOINTS:
    print('Server has no /status endpoint. Falling back to /results.')
    print('If that also returns processing, the run is opaque - need to look at Flask logs.')
    r = requests.get(f'{BASE_URL}/results/{LOAD_ID}', timeout=30)
    print(json.dumps(r.json(), indent=2)[:2000])
else:
    r = requests.get(f'{BASE_URL}/status/{LOAD_ID}', timeout=30)
    if r.status_code == 404:
        print(f'No status blob for load_id={LOAD_ID}')
        print('Either the load was never submitted, or the load_id is wrong.')
        status = None
    else:
        status = r.json()
        print('Status snapshot:')
        print(f'  status:         {status.get("status")}')
        print(f'  build_version:  {status.get("build_version")}')
        print(f'  started_at:     {status.get("started_at")}')
        print(f'  updated_at:     {status.get("updated_at")}')
        print(f'  completed_at:   {status.get("completed_at")}')
        print(f'  n_plans_total:  {status.get("n_plans_total")}')
        print(f'  n_plans_done:   {status.get("n_plans_done")}')
        print(f'  n_plans_failed: {status.get("n_plans_failed")}')


## 3. Per-plan breakdown

In [ ]:
if status and status.get('plans'):
    plans = status['plans']
    counts_by_status = {}
    for info in plans.values():
        s = info.get('status', 'unknown')
        counts_by_status[s] = counts_by_status.get(s, 0) + 1

    print('Plan status distribution:')
    for s, n in sorted(counts_by_status.items()):
        print(f'  {s:>12}: {n}')

    # Show plans currently processing or recently failed
    print()
    print('Plans currently processing (if any):')
    proc_plans = [(name, info) for name, info in plans.items()
                  if info.get('status') == 'processing']
    if proc_plans:
        for name, info in proc_plans:
            print(f'  {name}  started_at={info.get("started_at")}')
    else:
        print('  (none)')

    print()
    print('Failed plans (if any):')
    failed = [(name, info) for name, info in plans.items()
              if info.get('status') == 'failed']
    if failed:
        for name, info in failed[:20]:
            err = info.get('error', '')[:120]
            print(f'  {name}  ({info.get("elapsed_s", "?")}s): {err}')
        if len(failed) > 20:
            print(f'  ... and {len(failed) - 20} more')
    else:
        print('  (none)')

    # Top 10 slowest completed plans (useful for spotting outliers)
    print()
    done_plans = [(name, info) for name, info in plans.items()
                  if info.get('status') == 'done']
    if done_plans:
        slowest = sorted(done_plans, key=lambda x: x[1].get('elapsed_s', 0), reverse=True)[:10]
        print('Slowest completed plans:')
        for name, info in slowest:
            print(f'  {info.get("elapsed_s", "?"):>6.1f}s  {info.get("rows", 0):>5} rows  {name}')


## 4. Verdict

In [ ]:
if not status:
    print('No status to diagnose. Either the load was never submitted')
    print('via the new code path, or the load_id is wrong.')
else:
    # When was the status last updated?
    updated_at_str = status.get('updated_at')
    started_at_str = status.get('started_at')

    if updated_at_str:
        updated_at = datetime.fromisoformat(updated_at_str.replace('Z', '+00:00'))
        seconds_since_update = (datetime.now(timezone.utc) - updated_at).total_seconds()
    else:
        seconds_since_update = None

    if started_at_str:
        started_at = datetime.fromisoformat(started_at_str.replace('Z', '+00:00'))
        total_runtime = (datetime.now(timezone.utc) - started_at).total_seconds()
    else:
        total_runtime = None

    print(f'Total runtime so far: {total_runtime:.0f}s' if total_runtime else 'No start time')
    print(f'Time since last status update: {seconds_since_update:.0f}s' if seconds_since_update else '')
    print()

    n_done = status.get('n_plans_done', 0)
    n_total = status.get('n_plans_total', 0)
    n_failed = status.get('n_plans_failed', 0)

    if status.get('status') == 'success':
        print('VERDICT: complete')
        print(f'  All {n_total} plans done. Check outbound/output_benefits_{LOAD_ID}.json')

    elif status.get('status') == 'partial':
        print('VERDICT: partial complete')
        print(f'  {n_done}/{n_total} plans done, {n_failed} failed.')
        print(f'  Re-run with same LOAD_ID to retry pending plans.')

    elif seconds_since_update and seconds_since_update > 180:
        # No status update in 3 minutes - worker probably died
        print('VERDICT: worker likely DIED')
        print(f'  Status has not been updated for {seconds_since_update:.0f}s')
        print(f'  But status still says "processing" with {n_done}/{n_total} done.')
        print(f'  The Flask process probably restarted, killing the background thread.')
        print(f'  Action: re-run the batch script or POST /save again with same load.')
        print(f'  It will resume from checkpoint - the {n_done} completed plans are reused.')

    elif n_done == 0 and total_runtime and total_runtime > 300:
        print('VERDICT: stuck on FIRST plan')
        print(f'  {total_runtime:.0f}s elapsed with zero plans done.')
        print(f'  Either the first plan is pathologically large, the sweeper is stuck')
        print(f'  on it, or there is an Azure OpenAI throttle / connectivity issue.')
        print(f'  Action: look at Flask terminal logs for the per-benefit timing.')
        print(f'  If a single benefit call has been running >5 min, kill and restart.')

    elif n_done < n_total:
        avg_per_plan = total_runtime / n_done if n_done else None
        plans_remaining = n_total - n_done
        if avg_per_plan:
            est_remaining = plans_remaining * avg_per_plan
            print('VERDICT: making progress')
            print(f'  {n_done}/{n_total} plans done at avg {avg_per_plan:.1f}s/plan')
            print(f'  Estimated time remaining: {est_remaining:.0f}s ({est_remaining/60:.1f} min)')
            if est_remaining > 1200:
                print()
                print(f'  This load will take a while. Options:')
                print(f'    1. Be patient - it WILL finish, no need to keep polling')
                print(f'    2. Switch to batch script: LOAD_ID={LOAD_ID} python run_benefits_creation.py')
                print(f'       (runs to completion, no client timeout)')
                print(f'    3. Use MAX_PLANS_THIS_RUN to chunk into smaller passes')
        else:
            print('VERDICT: status unclear - new run with no plans complete yet')


## 5. (Optional) Watch live progress

Uncomment the cell below to poll once a minute and watch the plan counts tick up.
Stop the cell whenever you want - it does NOT affect the server.

In [ ]:
# import time
# from IPython.display import clear_output
#
# for i in range(60):  # poll for up to an hour
#     try:
#         r = requests.get(f'{BASE_URL}/status/{LOAD_ID}', timeout=15)
#         s = r.json()
#         clear_output(wait=True)
#         print(f'Poll #{i+1}')
#         print(f'  status:          {s.get("status")}')
#         print(f'  plans done:      {s.get("n_plans_done")} / {s.get("n_plans_total")}')
#         print(f'  plans failed:    {s.get("n_plans_failed")}')
#         print(f'  last updated:    {s.get("updated_at")}')
#         if s.get('status') in ('success', 'partial', 'error'):
#             print('Run finished.')
#             break
#     except Exception as e:
#         print(f'Poll error: {e}')
#     time.sleep(60)
